# Chapter 7 &mdash; The Language of an NFA: Eclose&ndash;Move&ndash;Eclose

**Concept 7 of the Chapter 7 decomposition:** *The Language of an NFA: $\hat{\delta}$ via $Eclosure$–$\delta$–$Eclosure$*

$\hat{\delta}(q,\varepsilon)=Eclosure(q)$; each symbol is Eclose, move, Eclose.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter7/Concept-Delta-Hat-Via-Eclosure/Concept-Delta-Hat-Via-Eclosure.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.AnimateNFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


$\hat{\delta}$ for an NFA takes a **set** to a **set**:

* **basis:** $\hat{\delta}(S,\varepsilon) = Eclosure(S)$;
* **step:** $\hat{\delta}(S, aw) = \hat{\delta}\big(Eclosure(\delta(Eclosure(S),a)),\ w\big)$.

Read the step as the three-phase rhythm **Eclose &ndash; move &ndash; Eclose**: settle where free
moves take you, consume one symbol, settle again.

Acceptance: $w \in L(N)$ iff $\hat{\delta}(Q_0, w) \cap F \neq \emptyset$. Note the
basis is *not* $S$ &mdash; forgetting the initial $Eclosure$ is the classic bug.

## 2. Definitions

### A machine where the initial $Eclosure$ matters

In [ ]:
N = md2mc('''NFA
I : '' -> A
A : '' -> F          !! epsilon alone reaches a final state
A : 0 -> A
''')

### $\hat{\delta}$, with the three-phase step

In [ ]:
def dhat(N, S, w):
    cur = Eclosure(N, set(S))                       # basis
    for a in w:
        moved = {t for q in cur for t in step_nfa(N, q, a)}
        cur = Eclosure(N, moved)                    # Eclose - move - Eclose
    return cur

def acc(N, w): return bool(dhat(N, N["Q0"], w) & N["F"])

### and the buggy version that skips the initial $Eclosure$

In [ ]:
def dhat_buggy(N, S, w):
    cur = set(S)
    for a in w:
        cur = Eclosure(N, {t for q in cur for t in step_nfa(N, q, a)})
    return cur

## 3. Tests

The basis case is $Eclosure$, not the set itself.

In [ ]:
print("dhat(N, Q0, '')   =", sorted(dhat(N, N["Q0"], '')))
print("Q0 itself         =", sorted(N["Q0"]))
assert dhat(N, N["Q0"], '') == Eclosure(N, N["Q0"])
print("\naccepts epsilon?", acc(N, ''), " -- only because of the initial Eclosure")
assert acc(N, '')

Skipping it gets $\varepsilon$ wrong &mdash; the classic bug, made visible.

In [ ]:
buggy = bool(dhat_buggy(N, N["Q0"], '') & N["F"])
print("correct : accepts '' =", acc(N, ''))
print("buggy   : accepts '' =", buggy)
assert acc(N, '') and not buggy

Our $\hat{\delta}$ agrees with `accepts_nfa` everywhere.

In [ ]:
from itertools import product
sig = sorted(N["Sigma"])                       # this machine's alphabet is just {'0'}
strs = [''.join(p) for k in range(10) for p in product(sig, repeat=k)]
assert all(acc(N, s) == accepts_nfa(N, s) for s in strs)
print("alphabet %s; agrees with accepts_nfa on all %d strings up to length 9"
      % (sig, len(strs)))

And Jove's `run_nfa` is the same function under another name.

In [ ]:
for s in ['', '0', '00', '000']:
    print("%-6r dhat=%-16s run_nfa=%s"
          % (s, sorted(dhat(N, N["Q0"], s)), sorted(run_nfa(N, N["Q0"], s))))
assert all(dhat(N, N["Q0"], s) == run_nfa(N, N["Q0"], s)
           for s in ['', '0', '00', '000'])

## 4. Animation

Eclose, move, Eclose &mdash; the rhythm the animation makes visible.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateNFA import *
AnimateNFA(N, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. What does `accepts_nfa(N, s, chatty=True)` print? Run it.
2. Give a machine where the *final* $Eclosure$ of a step matters but the initial one does not.
3. Write $\hat{\delta}$ recursively rather than as a loop.

In [ ]:
# Your work for the exercises above.